In [ ]:
import sys
sys.path.append('/data/users/quentin/package_paper/')
import scClone2DR as sccdr
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy
import pickle
import torch
from tqdm import tqdm
import os
import plotly.io as pio
import h5py
np.float_ = np.float64

rootpath = '/data/users/quentin/package_paper/experiments/paper_results'

COHORT = 'melanoma'
gene_set_collections = ['gene','c6','hallmarks', 'c2_pid']
cohort2clonemodes = {'melanoma': ['scatrex', 'phenograph'], 'aml':['phenograph']}
gene_set_collection = gene_set_collections[2]
clonemode = cohort2clonemodes[COHORT][1]
mode_features = 'metacells_{0}_{1}'.format(gene_set_collection, clonemode)

directory = os.path.join(rootpath,COHORT+"_baselines")
if not os.path.exists(directory):
    os.makedirs(directory)
pathsave = os.path.join(rootpath,COHORT+"_baselines",gene_set_collection)
if not os.path.exists(pathsave):
    os.makedirs(pathsave)
    
if COHORT=='melanoma':
    path_rna = '/data/users/04_share_reanalysis_results/melanoma_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
    path_fastdrug = '/data/users/04_share_reanalysis_results/melanoma_2025/MEL_CNN_abundances_no_plate_effect_correction.csv'
    concentration_DMSO = '100'
    concentration_drug = '5'
    path_info_cohort = None
elif COHORT=='aml':
    concentration_DMSO = '200'
    concentration_drug = '10'
    path_rna = '/data/users/04_share_reanalysis_results/aml_2025/02_atypical_removed_preprocessing/{0}/'.format(mode_features)
    path_fastdrug = '/data/users/04_share_reanalysis_results/01_aml/AML_PCY_cell_numbers_no_plate_effect_correction.csv'
    path_info_cohort = '/data/users/04_share_reanalysis_results/01_aml/2024-08-15_aml_overview_scRNA.tsv'
model = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, path_info_cohort=path_info_cohort, type_guide="lowrank_MVN")
data_ref = model.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)


if True:
    idxs_train = [i for i in range(int(data_ref['N']))]
    idxs_test = []
    data_train, data_test, sample_names_train, sample_names_test = model.get_real_data_split(idxs_train, idxs_test)
else:
    with open(os.path.join(rootpath,'{0}/{1}_{2}/params_svi.pkl'.format(COHORT, gene_set_collection, clonemode)), 'rb') as handle:
        params_svi = pickle.load(handle)

# Factorization machine

In [ ]:
modelFM = sccdr.models.FM_sc(model.cluster2clonelabel, model.clonelabel2cat)
modelFM.train(data_train)
modelFM.eval(data_train)
li_weights = modelFM.get_local_importance_weights(data_train)

In [ ]:
li_weights = li_weights.detach().numpy()
latent_dim = li_weights.shape[-1]
for j in range(latent_dim):
    for d in range(model.D):
        li_weights[d,:,:,j][~(data_train['masks']['RNA'])] = float('nan')
all_local_importances = li_weights
with h5py.File(os.path.join(pathsave,'local_importance.h5'), 'w') as f:
    # Create a dataset
    dset = f.create_dataset('local_importance_mean', data=all_local_importances)

    column_names = {
    'dim2_subclones': [i for i in range(data_train['Kmax'])],
    'dim3_samples': sample_names_train, #np.char.encode(sample_names_test, 'ascii'),
    'dim4_dimensions': model.feature_names,
    'dim1_drugs':  model.FD.selected_drugs
    }

    for dim, labels in column_names.items():
        dset.attrs[dim] = labels

In [ ]:
postmean = modelFM.pi.numpy()
for d in range(data_train['D']):
    postmean[d,:,:][~(data_train['masks']['RNA'])] = float('nan')
    

with h5py.File(os.path.join(pathsave,'factorization_machine_survival_probabilities.h5'), 'w') as f:
    # Create a dataset
    dset = f.create_dataset('survival_probabilities', data=postmean)
    column_names = {
    'dim2_subclones': [i for i in range(data_train['Kmax'])],
    'dim3_samples': sample_names_train, # np.char.encode(sample_names, 'ascii'),
    'dim1_drugs': model.FD.selected_drugs
    }

    for dim, labels in column_names.items():
        dset.attrs[dim] = labels


In [ ]:
plt.scatter(modelFM.results['fold_change_obs'].flatten(), modelFM.results['fold_change_pred'].flatten())


# Base

In [ ]:
model_base = sccdr.models.scClone2DR(path_fastdrug=path_fastdrug, path_rna=path_rna, type_guide="lowrank_MVN", rank=10)
data_base_ref = model_base.get_real_data(concentration_DMSO=concentration_DMSO, concentration_drug=concentration_drug)
del data_base_ref
model_base.data['X'] = torch.zeros(model_base.data['X'].shape)
data_base_train, _, sample_names_train, sample_names_test = model_base.get_real_data_split(idxs_train, idxs_test)
params_svi_base = model_base.train(data_base_train, penalty_l1=0.1, penalty_l2=0.1 , n_steps=200)

In [ ]:
model_base.compute_all_stats4bulk_or_bimodal(data_train)

# Bulk

In [ ]:
model_bulk = sccdr.models.scClone2DR()
databulk = model_bulk.get_bulk_from_real_data(data_train, model.cluster2cat, model.cat2clusters)
model_bulk.data = databulk
model_bulk.D = model.D
model_bulk.R = model.R
model_bulk.C = model.C
model_bulk.N = model.N
model_bulk.latent_dim = model.latent_dim
model_bulk.feature_names = model.feature_names
model_bulk.sample_names = model.sample_names
model_bulk.rank = model.rank
model_bulk.FD = model.FD
databulk, _ = model_bulk.add_design_preassay_bulk(databulk, {}, model_bulk.sample_names, [])
model_bulk.data = databulk

data_bulk_train, _, sample_names_train, sample_names_test = model_bulk.get_real_data_split(idxs_train, idxs_test)
params_svi_bulk = model_bulk.train(data_bulk_train, penalty_l1=0.1, penalty_l2=0.1 , n_steps=200)

In [ ]:
torch.sum(torch.isnan(model_bimodal.data['X']))

In [ ]:
posterior_mean_params = model_bulk.sampling_from_posterior(data_bulk_train, pathsave, params=params_svi_bulk, nb_ites=30, sample_names=model_bulk.sample_names, , name_model='bulk_')

# Bimodal

In [ ]:
model_bimodal = sccdr.models.scClone2DR()
databimodal = model_bimodal.get_bimodal_from_real_data(data_train, model.cluster2cat, model.cat2clusters)
model_bimodal.data = databimodal
model_bimodal.D = model.D
model_bimodal.R = model.R
model_bimodal.C = model.C
model_bimodal.N = model.N
model_bimodal.latent_dim = model.latent_dim
model_bimodal.feature_names = model.feature_names
model_bimodal.sample_names = model.sample_names
model_bimodal.rank = model.rank
model_bimodal.FD = model.FD
databimodal, _ = model_bimodal.add_design_preassay(databimodal, {}, model_bimodal.sample_names, [])
model_bimodal.data = databimodal

data_bimodal_train, _, sample_names_train, sample_names_test = model_bimodal.get_real_data_split(idxs_train, idxs_test)
params_svi_bimodal = model_bimodal.train(data_bimodal_train, penalty_l1=0.1, penalty_l2=0.1 , n_steps=200)

#posterior_mean_params = model_bimodal.sampling_from_posterior(data_bimodal_train, pathsave, params=params_svi_bimodal, nb_ites=30, sample_names=model_bimodal.sample_names, name_model='bimodal_')
#with open(os.path.join(rootpath,'{0}/{1}_{2}{3}/{4}posterior_mean_params.pkl'.format(COHORT+"_baselines", gene_set_collection, clonemode, addi, 'bimodal_')), 'wb') as handle:
#    pickle.dump(posterior_mean_params, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
import pyro
from torch.distributions import constraints
from pyro import poutine
import pyro.distributions as dist
import torch.distributions as tdist
theta_rna = None
fixed_proportions = False


data_prior = data_bimodal_train.copy()

Kmax = data_prior['Kmax']
R,D,Ndrug = data_prior['n_r'].shape
C,N = data_prior['n_c'].shape

masks_train = data_prior['masks']
dim = data_prior['X'].shape[-1]
beta = pyro.param('beta', torch.zeros((D,dim)))


params_pi = {"beta":beta}
for clonelabel in model_bimodal.clonelabels:
    params_pi["offset_{0}".format(clonelabel)] = pyro.param("offset_{0}".format(clonelabel), torch.zeros(D))
if data_prior['single_cell_features']:
    var_fs = {}
    for clonelabel in model_bimodal.clonelabels:
        var_fs[clonelabel] = pyro.param("f_{0}".format(clonelabel), torch.tensor(0.1), constraints.positive)  
        params_pi['gamma_{0}'.format(clonelabel)] = pyro.sample("gamma_{0}".format(clonelabel), dist.Normal(torch.zeros(dim), var_fs[clonelabel]*torch.ones(dim)).to_event(1))
    pi = model_bimodal.compute_survival_probas_single_cell_features(data_prior, params_pi)
else:
    pi = model_bimodal.compute_survival_probas_subclone_features(data_prior, params_pi)


if theta_rna is None:
    theta_rna = pyro.param('theta_rna', torch.tensor(40), constraints.positive)

try:
    weights = data_prior['weights']
except:
    weights = torch.ones(N)

try:
    weights_ND = data_prior['weights_ND']
except:
    weights_ND = torch.ones((D,N))


if model_bimodal.mode_nu == "noise_correction":
    dim_c = data_prior['X_nu_control'].shape[2]
    beta_control = pyro.param('beta_control', torch.zeros(dim_c))

with pyro.plate('samples', N):
    with pyro.poutine.scale(scale=weights):
        if fixed_proportions:
            proportions = (data_prior['n_rna'] / torch.tile(torch.sum(data_prior['n_rna'], dim=0).reshape(1,N), (Kmax,1)))
        else:
            proportions = pyro.param('proportions', data_prior['ini_proportions'].clone(), constraints.simplex).T


        # OVERDISPERSION PARAMETER (One for each patient)
        if model_bimodal.mode_theta=='equal':
            theta_fd = theta_rna
        elif model_bimodal.mode_theta=='shared':
            theta_fd = pyro.param('theta_fd', torch.tensor(1.), constraints.positive)
        elif 'not shared' in model_bimodal.mode_theta:
            theta_fd = pyro.param('theta_fd', 40.*torch.ones(N), constraints.positive)

        # RNA DATA
        # When calculating the log probability Multinomial distribution ignores total counts argument (it calculates it from the values). 
        # Since you are not sampling from the Multinomial distribution here, we can safely just set total_count to an arbitrary value
        if not(data_prior['n_rna'] is None): # for the bulk model
            if model_bimodal.mode_theta == "no_overdispersion":
                n_rna = pyro.sample('n_rna', dist.Multinomial(100000, proportions.T), obs=data_prior['n_rna'].T)
            elif model_bimodal.mode_theta in ['equal', 'shared', 'not shared decoupled']:
                n_rna = pyro.sample('n_rna', dist.DirichletMultinomial((theta_rna * proportions).T, torch.sum(data_prior['n_rna'], dim=0)), obs=data_prior['n_rna'].T)
            else:
                n_rna = pyro.sample('n_rna', dist.DirichletMultinomial(( theta_rna * theta_fd * proportions).T, torch.sum(data_prior['n_rna'], dim=0)), obs=data_prior['n_rna'].T)

        # CONTROL WELLS
        with pyro.plate('controls',C), poutine.mask(mask=masks_train['C']):   

            if model_bimodal.mode_nu=="fixed":
                nu_tumor_over_nu_healthy = torch.tensor(1)
                nu_healthy = 1./(1+nu_tumor_over_nu_healthy)

            elif model_bimodal.mode_nu=="noise_correction":
                nu_tumor_over_nu_healthy = torch.exp(data_prior['X_nu_control'] @ beta_control)               
                nu_healthy = 1./(1+nu_tumor_over_nu_healthy)

            nu_tumor = 1-nu_healthy

            if model_bimodal.mode_theta == "no_overdispersion":
                n0_c = pyro.sample('n0_c', dist.Binomial(100000, (proportions[0,:]).unsqueeze(0).repeat(C, 1)*nu_healthy), obs=data_prior['n0_c'])
            else:
                n0_c = pyro.sample('n0_c', dist.BetaBinomial((theta_fd*torch.sum(proportions[model_bimodal.cat2clusters['healthy'],:], dim=0)).unsqueeze(0).repeat(C, 1)*nu_healthy, (theta_fd*torch.sum(proportions[model_bimodal.cat2clusters['tumor'],:], dim=0)).unsqueeze(0).repeat(C, 1)* (1 - nu_healthy), data_prior['n_c']), obs=data_prior['n0_c'])
                
with pyro.plate('samples_drug', Ndrug):
            
    # WELLS WITH DRUG  
    with pyro.plate('drugs',D):
        with pyro.plate('replicates',R), poutine.mask(mask=masks_train['R']):
            if model_bimodal.mode_nu=="fixed":
                nu_tumor_over_nu_healthy = torch.tensor(1)
                nu_healthy_drug = 1./(1+nu_tumor_over_nu_healthy)
            elif model_bimodal.mode_nu=="noise_correction":
                nu_tumor_over_nu_healthy = torch.exp(data_prior['X_nu_drug'] @ beta_control)               
                nu_healthy_drug = 1./(1+nu_tumor_over_nu_healthy)
            nu_tumor_drug = 1-nu_healthy_drug

            if 'not' in model_bimodal.mode_theta:
                theta_fd_mode = theta_fd[:Ndrug]
            else:
                theta_fd_mode = theta_fd
                
            assert torch.sum(torch.isnan(theta_fd_mode))==0
            assert torch.sum(torch.isnan(torch.sum(proportions[model_bimodal.cat2clusters['healthy'],:Ndrug], dim=0)))==0
            assert torch.sum(torch.isnan(pi[:,model_bimodal.cat2clusters['healthy'],:]))==0
            temp = (theta_fd_mode*torch.sum(proportions[model_bimodal.cat2clusters['healthy'],:Ndrug].unsqueeze(0).repeat(D, 1,1)*pi[:,model_bimodal.cat2clusters['healthy'],:], dim=1)).unsqueeze(0).repeat(R,1,1)*nu_healthy_drug
            if model_bimodal.mode_theta == "no_overdispersion":
                n0_r = pyro.sample('n0_r', dist.Binomial(100000, (torch.sum(proportions[model_bimodal.cat2clusters['healthy'],:Ndrug].unsqueeze(0).repeat(D, 1,1)*pi[:,model_bimodal.cat2clusters['healthy'],:], dim=1)).unsqueeze(0).repeat(R,1,1)*nu_healthy_drug), obs=data_prior['n0_r'])
            else:
                n0_r = pyro.sample('n0_r', dist.BetaBinomial((theta_fd_mode*torch.sum(proportions[model_bimodal.cat2clusters['healthy'],:Ndrug].unsqueeze(0).repeat(D, 1,1)*pi[:,model_bimodal.cat2clusters['healthy'],:], dim=1)).unsqueeze(0).repeat(R,1,1)*nu_healthy_drug, (theta_fd_mode*torch.sum(proportions[model_bimodal.cat2clusters['tumor'],:Ndrug].unsqueeze(0).repeat(D, 1,1)*pi[:,model_bimodal.cat2clusters['tumor'],:], dim=1)).unsqueeze(0).repeat(R, 1, 1)* (1 - nu_healthy_drug),data_prior['n_r']), obs=data_prior['n0_r'])
            frac_r = 1. - n0_r / data_prior['n_r']

In [ ]:
print(torch.sum(torch.isnan(databimodal['X'][:,:,:,0])))
print(torch.sum(~(model_bimodal.data['masks']['SingleCell'][:,:,:])))

mask = model_bimodal.data['X']
mask.shape, model_bimodal.N, model_bimodal.data['masks']['SingleCell'].shape

In [ ]:
model_bimodal.data['masks']['SingleCell'][~torch.isnan(model_bimodal.data['X'][:,:,:,0])]

In [ ]:
data_train['X'].shape

# Neural Network

In [ ]:
Kmax, N, D, latent_dim = data_train['X'].shape
N = data_train['n_r'].shape[2]
D = data_train['D']
modelNN = sccdr.models.NN(modelscClone2DR.cluster2clonelabel, modelscClone2DR.clonelabel2cat)
modelNN.train(data_train, data_ref['beta'])
modelNN.eval(data_ref, true_params = {'pi': data_ref['pi'], 'beta':data_ref['beta']})

modelNN_trueprops = sccdr.models.NN(modelscClone2DR.cluster2clonelabel, modelscClone2DR.clonelabel2cat, use_true_proportions=True)
modelNN_trueprops.train(data_train, data_ref['beta'])
modelNN_trueprops.eval(data_ref, true_params = {'pi': data_ref['pi'], 'beta':data_ref['beta']})

# Visualizing results

In [ ]:
res = {}
res['us'] = modelscClone2DR.results
res['base'] = model_base.results
res['bulk'] = model_bulk.results
res['bimodal'] = model_bimodal.results
res['fm'] = modelFM.results
#res['fm_true_props'] = modelFM_trueprops.results
res['nn'] = modelNN.results
#res['nn_true_props'] = modelNN_trueprops.results
import seaborn as sns
colors_models = sns.color_palette('Set2')

#models = ['us', 'base', 'bimodal','bulk','fm','fm_true_props','nn','nn_true_props']
models = ['us', 'base', 'bimodal','bulk','fm','nn']
model2name = {'us':'scClone2DR', 'base':'Baseline', 'bimodal':'Dual bulk','bulk':'Bulk','fm':'FM','nn':'NN', 
              'fm_true_props': 'FM true props','nn_true_props':'NN true props'}

In [ ]:
for m in models:
    x = np.abs(1.-res[m]['true_drug_effects'].numpy())
    argidxs = np.argsort(x)
    argidxs_dec = np.flip(argidxs, axis=0)
    x = x[argidxs_dec]
    err = np.cumsum(np.abs(res[m]['true_drug_effects'].numpy()-res[m]['drug_effects'].numpy())[argidxs_dec])/np.cumsum(np.ones(len(x)))
    plt.plot(x, (err), label=model2name[m])
plt.legend(loc=2)
plt.ylabel('$L^1$ error on the drug effects', fontsize=14)
plt.xlabel('Threshold', fontsize=14)
plt.savefig('/data/users/quentin/final_package/experiments/simu_benchmark/L1err_scores.png', dpi=250, bbox_inches='tight')
plt.show()

In [ ]:
import seaborn as sns
dic = {'method': [], "drug scores": [], "statistic": []}
for m in models:
    dic['method'] += [model2name[m] for i in range(2*len(res[m]['drug_scores']))]
    dic['drug scores'] += list(np.array(res[m]['drug_scores']))
    dic['drug scores'] += list(np.array(res[m]['true_drug_scores']))
    dic['statistic'] += ['estimated' for i in range(len(res[m]['drug_scores']))]
    dic['statistic'] += ['ground truth' for i in range(len(res[m]['drug_scores']))]

df = pd.DataFrame(dic)
sns.set_theme(style='white')
sns.set_style("ticks", {"xtick.major.size": 12, "ytick.major.size": 12})

b = sns.violinplot(data=df, x="method", y="drug scores", hue="statistic", split=True, inner="quart", density_norm="width")

plt.xticks(rotation=40)
plt.legend(loc=2)
plt.ylabel('Drug scores', fontsize=14)
plt.xlabel('', fontsize=1)

plt.savefig('/data/users/quentin/final_package/experiments/simu_benchmark/drug_scores.png', dpi=250, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Initialize the dictionary
dic = {'method': [], "drug scores": [], "statistic": []}
mses = {}

# Loop through models and calculate MSE
for m in models:
    estimated_scores = np.array(res[m]['drug_scores'])
    true_scores = np.array(res[m]['true_drug_scores'])
    
    # Calculate MSE
    mse = np.mean((estimated_scores - true_scores) ** 2)
    mses[model2name[m]] = mse
    
    # Populate dictionary
    dic['method'] += [model2name[m] for i in range(2 * len(estimated_scores))]
    dic['drug scores'] += list(estimated_scores)
    dic['drug scores'] += list(true_scores)
    dic['statistic'] += ['estimated ' for i in range(len(estimated_scores))]
    dic['statistic'] += ['ground truth' for i in range(len(true_scores))]

# Convert to DataFrame
df = pd.DataFrame(dic)

# Set theme and style
sns.set_theme(style='white')
sns.set_style("ticks", {"xtick.major.size": 12, "ytick.major.size": 12})

# Create figure with two subplots
fig, (ax_violin, ax_bar) = plt.subplots(nrows=2, ncols=1, figsize=(10, 8), gridspec_kw={'height_ratios': [3, 1]})

# Violin plot on top
sns.violinplot(data=df, x="method", y="drug scores", hue="statistic", split=True, inner="quart", density_norm="width", ax=ax_violin)
ax_violin.set_ylabel('Drug Scores', fontsize=19)
ax_violin.set_xlabel('')
ax_violin.legend(loc=1, fontsize=19,ncol=2)
ax_violin.tick_params(axis='x', rotation=40)
ax_violin.set_xticks([])  # Remove the x-ticks
ax_violin.set_xticklabels([])  # Remove the x-tick labels

# MSE bar plot on the bottom
methods = list(mses.keys())
mse_values = list(mses.values())
ax_bar.bar(methods, mse_values, color='skyblue')
ax_bar.set_ylabel('MSE', fontsize=19)
#ax_bar.set_xlabel('Method', fontsize=18)
ax_bar.tick_params(axis='x', rotation=30, labelsize=19)

# Align the x-ticks and make sure both plots share the same x-axis
ax_bar.set_xticks(range(len(methods)))
ax_bar.set_xticklabels(methods)

# Save and show plot
plt.tight_layout()
plt.savefig('/data/users/quentin/final_package/experiments/simu_benchmark/drug_scores_mse.png', dpi=250, bbox_inches='tight')
plt.show()


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import collections

# https://stackoverflow.com/questions/70442958/seaborn-how-to-apply-custom-color-to-each-seaborn-violinplot

# Initialize the dictionary
dic = {'method': [], "drug scores": [], "statistic": [], 'ground_truth':[]}
mses = {}

# Loop through models and calculate MSE
for m in models:
    estimated_scores = np.array(res[m]['drug_scores'])
    true_scores = np.array(res[m]['true_drug_scores'])
    
    # Calculate MSE
    mse = np.mean((estimated_scores - true_scores) ** 2)
    mses[model2name[m]] = mse
    
    # Populate dictionary
    dic['method'] += [model2name[m] for i in range(2 * len(estimated_scores))]
    dic['drug scores'] += list(estimated_scores)
    dic['drug scores'] += list(true_scores)
    dic['ground_truth'] += [False for i in range(len(estimated_scores))]
    dic['ground_truth'] += [True for i in range(len(true_scores))]
    dic['statistic'] += ['estimated ' for i in range(len(estimated_scores))]
    dic['statistic'] += ['ground truth' for i in range(len(true_scores))]

# Convert to DataFrame
df = pd.DataFrame(dic)

# Set theme and style
sns.set_theme(style='white')
sns.set_style("ticks", {"xtick.major.size": 12, "ytick.major.size": 12})

# Create figure with two subplots
fig, (ax_violin, ax_bar) = plt.subplots(nrows=2, ncols=1, figsize=(10, 8), gridspec_kw={'height_ratios': [3, 1]})

# Violin plot on top
ax = sns.violinplot(data=df, x="method", y="drug scores", hue="statistic", split=True, inner="quart", density_norm="width", legend = False, ax=ax_violin)
for ind, violin in enumerate(ax.findobj(collections.PolyCollection)):
    rgb = colors_models[ind//2]
    if ind % 2 != 0:
        rgb = "gray"  # make white
    violin.set_facecolor(rgb)
ax_violin.plot([],[], linewidth=10, c="gray", label='Ground truth')
ax_violin.legend(fontsize=15)
ax_violin.set_ylabel('Drug Scores', fontsize=19)
ax_violin.set_xlabel('')
ax_violin.legend(loc=1, fontsize=19,ncol=2)
ax_violin.tick_params(axis='x', rotation=40)
ax_violin.set_xticks([])  # Remove the x-ticks
ax_violin.set_xticklabels([])  # Remove the x-tick labels

# MSE bar plot on the bottom
methods = list(mses.keys())
mse_values = list(mses.values())
ax_bar.bar(methods, mse_values, color=[colors_models[i] for i in range(len(models))])
ax_bar.set_ylabel('MSE', fontsize=19)
#ax_bar.set_xlabel('Method', fontsize=18)
ax_bar.tick_params(axis='x', rotation=30, labelsize=19)

# Align the x-ticks and make sure both plots share the same x-axis
ax_bar.set_xticks(range(len(methods)))
ax_bar.set_xticklabels(methods)

# Save and show plot
plt.tight_layout()
plt.savefig('/data/users/quentin/final_package/experiments/simu_benchmark/drug_scores_mse.png', dpi=250, bbox_inches='tight')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.gridspec import GridSpec


size_dots = 2
tit = ['This paper', 'Base model', 'Bimodal model', 'Bulk model',  'FM model', 'NN model', 'FM model (true props)', 'NN model (true props)']
# Sample data (replace with your actual data)
#colors = res['us']['colors']
colors = {m:colors_models[i] for i,m in enumerate((models))}

# Create subplots
fig = plt.figure(figsize=(8, 11))
gs = GridSpec(3, 2, width_ratios=[1, 1])

# Plot for model 1
ax00 = fig.add_subplot(gs[0, 0])
fc_true = res['us']['fold_change_true']
fc_pred = res['us']['fold_change_pred']
ax00.scatter(fc_true, fc_pred, color=colors['us'], s=size_dots)
ax00.set_title('{0}'.format(tit[0]))
ax00.set_xlabel('Fold change (ground truth)')
ax00.set_ylabel('Fold change (predicted)')
ax00.set_ylim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
# plt.hlines(0,xmin=lim_min,xmax=lim_max)
# plt.vlines(0,ymin=lim_min,ymax=lim_max)
# plt.xlim(lim_min,lim_max)
# plt.ylim(lim_min,lim_max)

# Plot for model 2
ax01 = fig.add_subplot(gs[0, 1])
fc_true = res['us']['fold_change_true']
fc_pred = res['base']['fold_change_pred']
ax01.scatter(fc_true, fc_pred, color=colors['base'], s=size_dots)
ax01.set_title('{0}'.format(tit[1]))
ax01.set_xlabel('Fold change (ground truth)')
ax01.set_ylabel('Fold change (predicted)')
ax01.set_ylim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
ax01.set_xlim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))

# Plot for model 3
ax10 = fig.add_subplot(gs[1, 0])
fc_true = res['us']['fold_change_true']
fc_pred = res['bimodal']['fold_change_pred']
ax10.scatter(fc_true, fc_pred, color=colors['bimodal'], s=size_dots)
ax10.set_title('{0}'.format(tit[2]))
ax10.set_xlabel('Fold change (ground truth)')
ax10.set_ylabel('Fold change (predicted)')
ax10.set_ylim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
ax10.set_xlim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))

# Plot for model 4
ax11 = fig.add_subplot(gs[1, 1])
fc_true = res['us']['fold_change_true']
fc_pred = res['bulk']['fold_change_pred']
ax11.scatter(fc_true, fc_pred, color=colors['bulk'], s=size_dots)
ax11.set_title('{0}'.format(tit[3]))
ax11.set_xlabel('Fold change (ground truth)')
ax11.set_ylabel('Fold change (predicted)')
ax11.set_ylim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
ax11.set_xlim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))


if False:
    # Plot for model 5
    ax20 = fig.add_subplot(gs[2, 0])
    fc_true = res['us']['fold_change_true']
    fc_pred = res['fm_true_props']['fold_change_pred']
    ax20.scatter(fc_true, fc_pred, color=colors['fm_true_props'])
    ax20.set_title('{0}'.format(tit[4]))
    ax20.set_xlabel('Fold change (ground truth)')
    ax20.set_ylabel('Fold change (predicted)')
    ax20.set_ylim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
    ax20.set_xlim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
    
    # Plot for model 6
    ax21 = fig.add_subplot(gs[2, 1])
    fc_true = res['us']['fold_change_true']
    fc_pred = res['nn_true_props']['fold_change_pred']
    ax21.scatter(fc_true, fc_pred, color=colors)
    ax21.set_title('{0}'.format(tit[5]))
    ax21.set_xlabel('Fold change (ground truth)')
    ax21.set_ylabel('Fold change (predicted)')
    ax21.set_ylim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
    ax21.set_xlim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))


# Create a colorbar
# Adjust layout
plt.tight_layout()

plt.savefig('/data/users/quentin/final_package/experiments/simu_benchmark/fold_change.png', dpi=250, bbox_inches='tight')

# Show the plot
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.gridspec import GridSpec
from sklearn.metrics import explained_variance_score


size_dots = 2
colors = {m:colors_models[i] for i,m in enumerate((models))}

custom_font_size = 20
custom_fontsize_labelsaxis = 16
fontsizelegend = 18
tit = ['scClone2DR', 'Base model', 'Bimodal model', 'Bulk model',  'FM model', 'NN model', 'FM model (true props)', 'NN model (true props)']
# Sample data (replace with your actual data)
#colors = res['us']['colors']

# Create subplots
fig = plt.figure(figsize=(12, 8))
gs = GridSpec(2, 3, width_ratios=[1, 1, 1])

# Plot for model 1
ax00 = fig.add_subplot(gs[0, 0])
fc_true = res['us']['fold_change_true']
fc_pred = res['us']['fold_change_pred']
corr = np.round(np.corrcoef(fc_true, fc_pred)[0,1]**2, 3)
rsquare = np.round(explained_variance_score(fc_true, fc_pred), 3)
ax00.scatter(fc_true, fc_pred, label='{0}'.format(np.round(corr,3)), s=size_dots, c=colors['us'])
ax00.set_title('{0}'.format(tit[0]), fontsize=custom_font_size)
ax00.set_xlabel('Fold change (ground truth)', fontsize=custom_fontsize_labelsaxis)
ax00.set_ylabel('Fold change (predicted)', fontsize=custom_fontsize_labelsaxis)
m = min([np.min(fc_pred),np.min(fc_true)])
ax00.set_ylim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
ax00.text(m+0.02,m+0.02, "Explained variance: {0}".format(corr,rsquare), fontsize=14)
#ax00.legend(title='Correlation', fontsize=fontsizelegend)
# plt.hlines(0,xmin=lim_min,xmax=lim_max)
# plt.vlines(0,ymin=lim_min,ymax=lim_max)
# plt.xlim(lim_min,lim_max)
# plt.ylim(lim_min,lim_max)

# Plot for model 2
ax01 = fig.add_subplot(gs[0, 1])
fc_true = res['us']['fold_change_true']
fc_pred = res['base']['fold_change_pred']
corr = np.round(np.corrcoef(fc_true, fc_pred)[0,1]**2, 3)
rsquare = np.round(explained_variance_score(fc_true, fc_pred), 3)
ax01.scatter(fc_true, fc_pred, label='{0}'.format(np.round(corr,3)), s=size_dots, c=colors['base'])
ax01.set_title('{0}'.format(tit[1]), fontsize=custom_font_size)
ax01.set_xlabel('Fold change (ground truth)', fontsize=custom_fontsize_labelsaxis)
ax01.set_ylabel('Fold change (predicted)', fontsize=custom_fontsize_labelsaxis)
m = min([np.min(fc_pred),np.min(fc_true)])
ax01.set_ylim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
ax01.set_xlim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
ax01.text(m+0.02,m+0.02, "Explained variance: {0}".format(corr,rsquare), fontsize=14)
#ax01.legend(title='Correlation', fontsize=fontsizelegend)

# Plot for model 3
ax10 = fig.add_subplot(gs[0, 2])
fc_true = res['us']['fold_change_true']
fc_pred = res['bimodal']['fold_change_pred']
corr = np.round(np.corrcoef(fc_true, fc_pred)[0,1]**2, 3)
rsquare = np.round(explained_variance_score(fc_true, fc_pred), 3)
ax10.scatter(fc_true, fc_pred, label='{0}'.format(np.round(corr,3)), s=size_dots, c=colors['bimodal'])
ax10.set_title('{0}'.format(tit[2]), fontsize=custom_font_size)
ax10.set_xlabel('Fold change (ground truth)', fontsize=custom_fontsize_labelsaxis)
ax10.set_ylabel('Fold change (predicted)', fontsize=custom_fontsize_labelsaxis)
m = min([np.min(fc_pred),np.min(fc_true)])
ax10.set_ylim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
ax10.set_xlim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
ax10.text(m+0.02,m+0.02, "Explained variance: {0}".format(corr,rsquare), fontsize=14)
#ax10.legend(title='Correlation', fontsize=fontsizelegend)

# Plot for model 4
ax11 = fig.add_subplot(gs[1, 0])
fc_true = res['us']['fold_change_true']
fc_pred = res['bulk']['fold_change_pred']
corr = np.round(np.corrcoef(fc_true, fc_pred)[0,1]**2, 3)
rsquare = np.round(explained_variance_score(fc_true, fc_pred), 3)
ax11.scatter(fc_true, fc_pred, label='{0}'.format(np.round(corr,3)), s=size_dots, c=colors['bulk'])
ax11.set_title('{0}'.format(tit[3]), fontsize=custom_font_size)
ax11.set_xlabel('Fold change (ground truth)', fontsize=custom_fontsize_labelsaxis)
ax11.set_ylabel('Fold change (predicted)', fontsize=custom_fontsize_labelsaxis)
m = min([np.min(fc_pred),np.min(fc_true)])
ax11.set_ylim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
ax11.set_xlim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
ax11.text(m+0.02,m+0.02, "Explained variance: {0}".format(corr,rsquare), fontsize=14)
#ax11.legend(title='Correlation', fontsize=fontsizelegend)


# Plot for model 5
ax12 = fig.add_subplot(gs[1, 1])
fc_true = res['us']['fold_change_true']
fc_pred = res['fm']['fold_change_pred']
corr = np.round(np.corrcoef(fc_true, fc_pred)[0,1]**2, 3)
rsquare = np.round(explained_variance_score(fc_true, fc_pred), 3)
ax12.scatter(fc_true, fc_pred, label='{0}'.format(np.round(corr,3)), s=size_dots, c=colors['fm'])
ax12.set_title('{0}'.format(tit[4]), fontsize=custom_font_size)
ax12.set_xlabel('Fold change (ground truth)', fontsize=custom_fontsize_labelsaxis)
ax12.set_ylabel('Fold change (predicted)', fontsize=custom_fontsize_labelsaxis)
m = min([np.min(fc_pred),np.min(fc_true)])
ax12.set_ylim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
ax12.set_xlim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
ax12.text(m+0.02,m+0.02, "Explained variance: {0}".format(corr,rsquare), fontsize=14)
#ax12.legend(title='Correlation', fontsize=fontsizelegend)

# Plot for model 6
ax02 = fig.add_subplot(gs[1, 2])
fc_true = res['us']['fold_change_true']
fc_pred = res['nn']['fold_change_pred']
corr = np.round(np.corrcoef(fc_true, fc_pred)[0,1]**2, 3)
rsquare = np.round(explained_variance_score(fc_true, fc_pred), 3)
ax02.scatter(fc_true, fc_pred, label='{0}'.format(np.round(corr,3)), s=size_dots, c=colors['nn'])
ax02.set_title('{0}'.format(tit[5]), fontsize=custom_font_size)
m = min([np.min(fc_pred),np.min(fc_true)])
ax02.set_xlabel('Fold change (ground truth)', fontsize=custom_fontsize_labelsaxis)
ax02.set_ylabel('Fold change (predicted)', fontsize=custom_fontsize_labelsaxis)
ax02.set_ylim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
ax02.set_xlim(min([np.min(fc_pred),np.min(fc_true)]), max([np.max(fc_pred),np.max(fc_true)]))
ax02.text(m+0.02,m+0.02, "Explained variance: {0}".format(corr,rsquare), fontsize=14)
#ax02.legend(title='Correlation', fontsize=fontsizelegend)

# Create a colorbar
# Adjust layout
plt.tight_layout()

plt.savefig('/data/users/quentin/final_package/experiments/simu_benchmark/fold_change.png', dpi=250, bbox_inches='tight')

# Show the plot
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.gridspec import GridSpec
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.colors import ListedColormap

size_dots = 2
colors = {m:colors_models[i] for i,m in enumerate((models))}

custom_font_size = 20
custom_fontsize_labelsaxis = 16
fontsizelegend = 18
tit = ['scClone2DR', 'Base model',  'FM model', 'NN model', 'FM model (true props)', 'NN model (true props)']
# Sample data (replace with your actual data)
#colors = res['us']['colors']

# Create subplots
fig = plt.figure(figsize=(12, 8))
gs = GridSpec(2, 2, width_ratios=[1, 1])

M = 0
pi_error = np.array(np.abs(res['us']['true_survival_probas']-res['us']['survival_probas']).reshape(-1))
M = np.max([M, np.max(pi_error)])
pi_error = np.array(np.abs(res['us']['true_survival_probas']-res['base']['survival_probas']).reshape(-1))
M = np.max([M, np.max(pi_error)])
pi_error = np.array(np.abs(res['us']['true_survival_probas']-res['fm']['pi']).reshape(-1))
M = np.max([M, np.max(pi_error)])
pi_error = np.array(np.abs(res['us']['true_survival_probas']-res['nn']['pi']).reshape(-1))
M = np.max([M, np.max(pi_error)])


# Plot for model 1
D, Kmax, N = res['us']['true_survival_probas'].shape
ax00 = fig.add_subplot(gs[0, 0])
props_true = (np.tile( (res['us']['true_proportions'].T).reshape(1,Kmax,N), (D,1,1))).reshape(-1)
pi_true = res['us']['true_survival_probas'].reshape(-1)
pi_error = np.array(np.abs(res['us']['true_survival_probas']-res['us']['survival_probas']).reshape(-1))

col = [tuple([el * (M-s)/M for el in colors['us']]) for s in pi_error]
ax00.scatter(props_true, pi_true,  s=size_dots, c=col)
norm = Normalize(vmin=0, vmax=M)
cmap = ListedColormap([tuple([el * (100-s)/100 for el in colors['us']]) for s in range(100)])

fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax00, label='Absolute error on survival probability')
ax00.set_title('{0}'.format(tit[0]), fontsize=custom_font_size)
ax00.set_xlabel('True proportion of subclones', fontsize=custom_fontsize_labelsaxis)
ax00.set_ylabel('Survival probability', fontsize=custom_fontsize_labelsaxis)
# plt.hlines(0,xmin=lim_min,xmax=lim_max)
# plt.vlines(0,ymin=lim_min,ymax=lim_max)
# plt.xlim(lim_min,lim_max)
# plt.ylim(lim_min,lim_max)

# Plot for model 2
ax01 = fig.add_subplot(gs[0, 1])
props_true = (np.tile( (res['base']['true_proportions'].T).reshape(1,Kmax,N), (D,1,1))).reshape(-1)
pi_true = res['base']['true_survival_probas'].reshape(-1)
pi_error = np.array(np.abs(res['base']['true_survival_probas']-res['base']['survival_probas']).reshape(-1))

col = [tuple([el * (M-s)/M for el in colors['base']]) for s in pi_error]
ax01.scatter(props_true, pi_true,  s=size_dots, c=col)
cmap = ListedColormap([tuple([el * (100-s)/100 for el in colors['base']]) for s in range(100)])
fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax01, label='Absolute error on survival probability')
ax01.set_title('{0}'.format(tit[1]), fontsize=custom_font_size)
ax01.set_xlabel('True proportion of subclones', fontsize=custom_fontsize_labelsaxis)
ax01.set_ylabel('Survival probability', fontsize=custom_fontsize_labelsaxis)

# # Plot for model 3
# ax01 = fig.add_subplot(gs[0, 2])
# props_true = (np.tile( (res['bimodal']['true_proportions'].T).reshape(1,Kmax,N), (D,1,1))).reshape(-1)
# pi_true = res['bimodal']['true_survival_probas'].reshape(-1)
# pi_error = np.array(np.abs(res['bimodal']['true_survival_probas']-res['bimodal']['survival_probas']).reshape(-1))

# col = [tuple([el * s/np.max(pi_error) for el in colors['bimodal']]) for s in pi_error]
# ax01.scatter(props_true, pi_true, label='{0}'.format(np.round(corr,3)), s=size_dots, c=col)
# ax01.set_title('{0}'.format(tit[0]), fontsize=custom_font_size)
# ax01.set_xlabel('True proportions of subclones', fontsize=custom_fontsize_labelsaxis)
# ax01.set_ylabel('Survival probability', fontsize=custom_fontsize_labelsaxis)

# # Plot for model 4
# ax11 = fig.add_subplot(gs[1, 0])
# props_true = (np.tile( (res['bulk']['true_proportions'].T).reshape(1,Kmax,N), (D,1,1))).reshape(-1)
# pi_true = res['bulk']['true_survival_probas'].reshape(-1)
# pi_error = np.array(np.abs(res['bulk']['true_survival_probas']-res['bulk']['survival_probas']).reshape(-1))

# col = [tuple([el * s/np.max(pi_error) for el in colors['bulk']]) for s in pi_error]
# ax11.scatter(props_true, pi_true, label='{0}'.format(np.round(corr,3)), s=size_dots, c=col)
# ax11.set_title('{0}'.format(tit[0]), fontsize=custom_font_size)
# ax11.set_xlabel('True proportions of subclones', fontsize=custom_fontsize_labelsaxis)
# ax11.set_ylabel('Survival probability', fontsize=custom_fontsize_labelsaxis)


# Plot for model 5
ax12 = fig.add_subplot(gs[1, 0])
props_true = (np.tile( (res['us']['true_proportions'].T).reshape(1,Kmax,N), (D,1,1))).reshape(-1)
pi_true = res['us']['true_survival_probas'].reshape(-1)
pi_error = np.array(np.abs(res['us']['true_survival_probas']-res['fm']['pi']).reshape(-1))

col = [tuple([el *  (M-s)/M for el in colors['fm']]) for s in pi_error]
cmap = ListedColormap([tuple([el * (100-s)/100 for el in colors['fm']]) for s in range(100)])
fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax12, label='Absolute error on survival probability')
ax12.scatter(props_true, pi_true,  s=size_dots, c=col)
ax12.set_title('{0}'.format(tit[2]), fontsize=custom_font_size)
ax12.set_xlabel('True proportion of subclones', fontsize=custom_fontsize_labelsaxis)
ax12.set_ylabel('Survival probability', fontsize=custom_fontsize_labelsaxis)


# Plot for model 6
ax02 = fig.add_subplot(gs[1, 1])
props_true = (np.tile( (res['us']['true_proportions'].T).reshape(1,Kmax,N), (D,1,1))).reshape(-1)
pi_true = res['us']['true_survival_probas'].reshape(-1)
pi_error = np.array(np.abs(res['us']['true_survival_probas']-res['nn']['pi']).reshape(-1))

col = [tuple([el *  (M-s)/M for el in colors['nn']]) for s in pi_error]
ax02.scatter(props_true, pi_true,  s=size_dots, c=col)
cmap = ListedColormap([tuple([el * (100-s)/100 for el in colors['nn']]) for s in range(100)])
fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax02, label='Absolute error on survival probability')
ax02.set_title('{0}'.format(tit[3]), fontsize=custom_font_size)
ax02.set_xlabel('True proportion of subclones', fontsize=custom_fontsize_labelsaxis)
ax02.set_ylabel('True survival probability', fontsize=custom_fontsize_labelsaxis)

# Create a colorbar
# Adjust layout
plt.tight_layout()

plt.savefig('/data/users/quentin/final_package/experiments/simu_benchmark/size_resistance_pi_errors.png', dpi=250, bbox_inches='tight')

# Show the plot
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.gridspec import GridSpec
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.colors import ListedColormap

size_dots = 2
colors = {m:colors_models[i] for i,m in enumerate((models))}

custom_font_size = 20
custom_fontsize_labelsaxis = 16
fontsizelegend = 18
tit = ['scClone2DR', 'Base model',  'FM model', 'NN model', 'FM model (true props)', 'NN model (true props)']
# Sample data (replace with your actual data)
#colors = res['us']['colors']

# Create subplots
fig = plt.figure(figsize=(12, 8))
gs = GridSpec(2, 2, width_ratios=[1, 1])

M = 0
pi_error = np.array(np.abs(res['us']['true_survival_probas']-res['us']['survival_probas']).reshape(-1))
M = np.max([M, np.max(pi_error)])
pi_error = np.array(np.abs(res['us']['true_survival_probas']-res['base']['survival_probas']).reshape(-1))
M = np.max([M, np.max(pi_error)])
pi_error = np.array(np.abs(res['us']['true_survival_probas']-res['fm']['pi']).reshape(-1))
M = np.max([M, np.max(pi_error)])
pi_error = np.array(np.abs(res['us']['true_survival_probas']-res['nn']['pi']).reshape(-1))
M = np.max([M, np.max(pi_error)])
m = 0


Mpi = np.max(np.array(res['us']['true_survival_probas']))
true_pi = np.array(res['us']['true_survival_probas']).reshape(-1)
mpi = np.min(np.array(res['us']['true_survival_probas']))

# Plot for model 1
D, Kmax, N = res['us']['true_survival_probas'].shape
ax00 = fig.add_subplot(gs[0, 0])
props_true = (np.tile( (res['us']['true_proportions'].T).reshape(1,Kmax,N), (D,1,1))).reshape(-1)
pi_true = res['us']['true_survival_probas'].reshape(-1)
pi_error = np.array(np.abs(res['us']['true_survival_probas']-res['us']['survival_probas']).reshape(-1))

col = [tuple([el * (Mpi-s+mpi)/Mpi for el in colors['us']]) for s in true_pi]
ax00.scatter(props_true, pi_error,  s=size_dots, c=col)
norm = Normalize(vmin=mpi, vmax=Mpi)
cmap = ListedColormap([tuple([el * (100-s)/100 for el in colors['us']]) for s in range(0,100)])

fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax00, label='True survival probability')
ax00.set_title('{0}'.format(tit[0]), fontsize=custom_font_size)
ax00.set_xlabel('True proportion of subclones', fontsize=custom_fontsize_labelsaxis)
ax00.set_ylabel('Absolute error\non survival probability', fontsize=custom_fontsize_labelsaxis)
# plt.hlines(0,xmin=lim_min,xmax=lim_max)
# plt.vlines(0,ymin=lim_min,ymax=lim_max)
# plt.xlim(lim_min,lim_max)
ax00.set_ylim(m,M)

# Plot for model 2
ax01 = fig.add_subplot(gs[0, 1])
props_true = (np.tile( (res['base']['true_proportions'].T).reshape(1,Kmax,N), (D,1,1))).reshape(-1)
pi_true = res['base']['true_survival_probas'].reshape(-1)
pi_error = np.array(np.abs(res['base']['true_survival_probas']-res['base']['survival_probas']).reshape(-1))

col = [tuple([el * (Mpi-s+mpi)/Mpi for el in colors['base']]) for s in true_pi]
ax01.scatter(props_true, pi_error,  s=size_dots, c=col)
cmap = ListedColormap([tuple([el * (100-s)/100 for el in colors['base']]) for s in range(100)])
fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax01, label='True survival probability')
ax01.set_title('{0}'.format(tit[1]), fontsize=custom_font_size)
ax01.set_xlabel('True proportion of subclones', fontsize=custom_fontsize_labelsaxis)
ax01.set_ylabel('Absolute error\non survival probability', fontsize=custom_fontsize_labelsaxis)
ax01.set_ylim(m,M)

# # Plot for model 3
# ax01 = fig.add_subplot(gs[0, 2])
# props_true = (np.tile( (res['bimodal']['true_proportions'].T).reshape(1,Kmax,N), (D,1,1))).reshape(-1)
# pi_true = res['bimodal']['true_survival_probas'].reshape(-1)
# pi_error = np.array(np.abs(res['bimodal']['true_survival_probas']-res['bimodal']['survival_probas']).reshape(-1))

# col = [tuple([el * s/np.max(pi_error) for el in colors['bimodal']]) for s in pi_error]
# ax01.scatter(props_true, pi_true, label='{0}'.format(np.round(corr,3)), s=size_dots, c=col)
# ax01.set_title('{0}'.format(tit[0]), fontsize=custom_font_size)
# ax01.set_xlabel('True proportions of subclones', fontsize=custom_fontsize_labelsaxis)
# ax01.set_ylabel('Survival probability', fontsize=custom_fontsize_labelsaxis)

# # Plot for model 4
# ax11 = fig.add_subplot(gs[1, 0])
# props_true = (np.tile( (res['bulk']['true_proportions'].T).reshape(1,Kmax,N), (D,1,1))).reshape(-1)
# pi_true = res['bulk']['true_survival_probas'].reshape(-1)
# pi_error = np.array(np.abs(res['bulk']['true_survival_probas']-res['bulk']['survival_probas']).reshape(-1))

# col = [tuple([el * s/np.max(pi_error) for el in colors['bulk']]) for s in pi_error]
# ax11.scatter(props_true, pi_true, label='{0}'.format(np.round(corr,3)), s=size_dots, c=col)
# ax11.set_title('{0}'.format(tit[0]), fontsize=custom_font_size)
# ax11.set_xlabel('True proportions of subclones', fontsize=custom_fontsize_labelsaxis)
# ax11.set_ylabel('Survival probability', fontsize=custom_fontsize_labelsaxis)


# Plot for model 5
ax12 = fig.add_subplot(gs[1, 0])
props_true = (np.tile( (res['us']['true_proportions'].T).reshape(1,Kmax,N), (D,1,1))).reshape(-1)
pi_true = res['us']['true_survival_probas'].reshape(-1)
pi_error = np.array(np.abs(res['us']['true_survival_probas']-res['fm']['pi']).reshape(-1))

col = [tuple([el *  (Mpi-s+mpi)/Mpi for el in colors['fm']]) for s in true_pi]
cmap = ListedColormap([tuple([el * (100-s)/100 for el in colors['fm']]) for s in range(100)])
fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax12, label='True survival probability')
ax12.scatter(props_true, pi_error,  s=size_dots, c=col)
ax12.set_title('{0}'.format(tit[2]), fontsize=custom_font_size)
ax12.set_xlabel('True proportion of subclones', fontsize=custom_fontsize_labelsaxis)
ax12.set_ylabel('Absolute error\non survival probability', fontsize=custom_fontsize_labelsaxis)
ax12.set_ylim(m,M)


# Plot for model 6
ax02 = fig.add_subplot(gs[1, 1])
props_true = (np.tile( (res['us']['true_proportions'].T).reshape(1,Kmax,N), (D,1,1))).reshape(-1)
pi_true = res['us']['true_survival_probas'].reshape(-1)
pi_error = np.array(np.abs(res['us']['true_survival_probas']-res['nn']['pi']).reshape(-1))

col = [tuple([el *  (Mpi-s+mpi)/Mpi for el in colors['nn']]) for s in true_pi]
ax02.scatter(props_true, pi_error,  s=size_dots, c=col)
cmap = ListedColormap([tuple([el * (100-s)/100 for el in colors['nn']]) for s in range(100)])
fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax02, label='True survival probability')
ax02.set_title('{0}'.format(tit[3]), fontsize=custom_font_size)
ax02.set_xlabel('True proportion of subclones', fontsize=custom_fontsize_labelsaxis)
ax02.set_ylabel('Absolute error\non survival probability', fontsize=custom_fontsize_labelsaxis)
ax02.set_ylim(m,M)

# Create a colorbar
# Adjust layout
plt.tight_layout()

plt.savefig('/data/users/quentin/final_package/experiments/simu_benchmark/size_resistance_pi_errors.png', dpi=250, bbox_inches='tight')

# Show the plot
plt.show()

In [ ]:
pi_error = np.array((np.abs(res['us']['true_survival_probas']-res['fm']['pi'])/res['us']['true_survival_probas']).reshape(-1))
plt.hist(pi_error)

In [ ]:
temp = res['fm']['pi']
for k in range(temp.shape[1]):
    temp[:,k,:] *= res['us']['true_survival_probas'][:,0,:] / temp[:,0,:]
pi_error = np.array((np.abs(res['us']['true_survival_probas']-temp)).reshape(-1))
plt.hist(pi_error)

In [ ]:
temp = res['us']['survival_probas']
for k in range(temp.shape[1]):
    temp[:,k,:] *= res['us']['true_survival_probas'][:,0,:] / temp[:,0,:]
pi_error = np.array((np.abs(res['us']['true_survival_probas']-temp)).reshape(-1))
plt.hist(pi_error)

In [ ]:
stats = ['KL_props','KL_survival_probas','L1err_overall_survival','L1err_drug_effects','spearman_drugs_avg','spearman_subclones_avg']

model_names = ['us', 'base', 'bimodal','bulk','fm','nn','fm_true_props','nn_true_props']

for stat in stats:
    res_str = ""
    for model_name in model_names:
        try:
            res_str += str(np.round(np.array(res[model_name][stat]),3))
        except:
            res_str += " \\xmark "
        res_str += " & "
    print(res_str)

In [ ]:
res['us'].keys()